# Take/Give self-play RL on a small model (Qwen2.5-0.5B-Instruct, Colab)

A self-contained, single-GPU version of this repo's experiment: two copies of one model talk, then each
privately chooses **TAKE** or **GIVE**. Each agent is *told* only its own payoff:

| you / other | GIVE  | TAKE  |
|-------------|-------|-------|
| **GIVE**    | 2 / 2 | 0 / 6 |
| **TAKE**    | 6 / 0 | 1 / 1 |

During training both agents get the **pooled** reward (the sum of both payoffs): Take/Give = 6 > Give/Give = 4 >
Take/Take = 2. Does RL teach the model to give?

How it differs from the main repo (Qwen3-8B / gpt-oss-20b on Tinker):
- **Model**: `Qwen/Qwen2.5-0.5B-Instruct` with LoRA, plain PyTorch + `transformers` + `peft`. No Tinker, no API keys.
- **No reasoning phase**: Qwen2.5 isn't a thinking model, so each turn is just the message (the decision turn
  may explain itself before the `DECISION:` line).
- **Shorter talks**: 3 messages per player by default (repo: 10), and short replies, to keep it cheap.
- **Same RL setup otherwise**: one game = one episode, the whole game from one seat trains as one sequence,
  GRPO-style advantages centred within a group of 8 games (16 trajectories), invalid decision = reward −1,
  one on-policy gradient step per batch.

**Runtime → Change runtime type → GPU** (a T4 works; L4/A100 are much faster and use bf16). Then Runtime → Run all.
Set `pooled_reward=False` in the config for the control run (each agent trained on its own payoff).

In [ ]:
%pip -q install "peft>=0.13" "transformers>=4.45" accelerate

In [ ]:
import contextlib, json, math, os, random, re, time
from collections import Counter
from dataclasses import asdict, dataclass

import torch
import torch.nn.functional as F


@dataclass
class Config:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    seed: int = 0
    # game
    max_messages: int = 3            # per player (repo default: 10)
    allow_early_end: bool = True     # both players ending a message with [END] back to back ends the talk
    pooled_reward: bool = True       # False = control: each agent trained on its own stated payoff
    invalid_decision_reward: float = -1.0
    # RL: 8 groups x 8 games = 64 games (128 trajectories) per step
    steps: int = 40
    groups_per_batch: int = 8
    games_per_group: int = 8
    normalize_adv: bool = True       # divide group-centred advantages by the group's std (GRPO)
    lr: float = 5e-5
    lora_rank: int = 16
    max_grad_norm: float = 1.0
    micro_batch: int = 8             # trajectories per forward/backward; lower it if you run out of memory
    # sampling (temperature 1, no top-k/top-p, so the samples are on-policy)
    temperature: float = 1.0
    max_new_tokens_msg: int = 80
    max_new_tokens_decision: int = 128
    gen_batch_size: int = 128
    eval_games: int = 64
    log_path: str = "runs/qwen05b"


cfg = Config()
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
os.makedirs(cfg.log_path, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
bf16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # Ampere+ (L4, A100); T4 uses fp32
dtype = torch.bfloat16 if bf16 else torch.float32
print(device, torch.cuda.get_device_name() if device == "cuda" else "", dtype)
if device == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU.")

## The game

Same rules and prompts as `pd_game/game.py` and `pd_game/prompts.py`. Agents are told only their own payoffs;
pooling is never mentioned and nothing says the other agent is a copy of themselves.

In [ ]:
GIVE, TAKE = "GIVE", "TAKE"
PAYOFFS = {(GIVE, GIVE): (2, 2), (TAKE, GIVE): (6, 0), (GIVE, TAKE): (0, 6), (TAKE, TAKE): (1, 1)}
END_MARK = "[END]"

EARLY_END = (f" Either of you can propose ending the conversation early by putting {END_MARK} at the end of a message; "
             f"it ends once both of you have done so in back-to-back messages.") if cfg.allow_early_end else ""
SYSTEM = f"""You are an AI agent playing a one-shot game with another AI agent.

How the game works:
1. First, you and the other agent talk by taking turns sending messages. Each of you can send at most {cfg.max_messages} messages.{EARLY_END}
2. When the conversation is over, each of you privately and independently chooses TAKE or GIVE. Neither of you sees the other's choice before making your own.
3. Your reward depends on both choices:
   - You GIVE and the other agent GIVES: you get 2, the other agent gets 2.
   - You TAKE and the other agent GIVES: you get 6, the other agent gets 0.
   - You GIVE and the other agent TAKES: you get 0, the other agent gets 6.
   - You both TAKE: you get 1, the other agent gets 1.

This is a single game; you will not meet this agent again.

When it is your turn to talk, reply with only the message you want to send. The other agent sees your messages, but never your private reasoning."""

OPENING_FIRST = "The conversation begins now. You send the first message."
OPENING_SECOND = "The conversation begins now. The other agent sends the first message."
CUT_OFF_PARTNER = "(The other agent's message was cut off and could not be delivered.)"
ENDED_EARLY = "You both agreed to end the conversation."
DECISION_PROMPT = (
    "The conversation is over. Now make your decision, privately and independently. The other agent is "
    "deciding at the same time and will not see your choice before making theirs.\n\n"
    "End your reply with exactly one line: DECISION: TAKE or DECISION: GIVE")
_DECISION_RE = re.compile(r"DECISION\s*[:\-]\s*[*_`\"']*\s*(TAKE|GIVE)\b", re.IGNORECASE)


def partner_says(text):
    return f"The other agent says:\n\n{text}" if text.strip() else "(The other agent sent an empty message.)"


def your_turn(left):
    return f"Your turn. You have {left} message{'' if left == 1 else 's'} left. Write your message to the other agent."


def parse_decision(text):
    m = _DECISION_RE.findall(text or "")
    if m:
        return m[-1].upper()
    bare = (text or "").strip().strip("*_`.\"'").upper()
    return bare if bare in (GIVE, TAKE) else None


class Seat:
    """One player's side of a game: its token sequence (= everything it has seen and said) and the loss mask."""
    def __init__(self):
        self.tokens, self.mask = [], []
        self.seen = 0       # transcript entries already shown to this seat
        self.sent = 0
        self.turns = 0


class Game:
    def __init__(self, first_speaker):
        self.first = first_speaker
        self.transcript = []                 # dicts: speaker, text, truncated
        self.choice = [None, None]           # GIVE / TAKE / None (unparseable)
        self.decision_text = [None, None]    # None until the player has decided
        self.talk_over = self.ended_early = False
        self.seats = [Seat(), Seat()]

    @property
    def current_speaker(self):
        return self.first if len(self.transcript) % 2 == 0 else 1 - self.first

    @property
    def done(self):
        return all(t is not None for t in self.decision_text)

    def user_turn(self, pid):
        """Text of the next user turn for `pid`: everything that happened since its last reply, plus the ask."""
        s = self.seats[pid]
        parts = [OPENING_FIRST if pid == self.first else OPENING_SECOND] if s.turns == 0 else []
        for u in self.transcript[s.seen:]:
            if u["speaker"] != pid:
                parts.append(CUT_OFF_PARTNER if u["truncated"] else partner_says(u["text"]))
        s.seen = len(self.transcript)
        if self.talk_over:
            parts += ([ENDED_EARLY] if self.ended_early else []) + [DECISION_PROMPT]
        else:
            parts.append(your_turn(cfg.max_messages - s.sent))
        return "\n\n".join(parts)

    def post(self, pid, text, truncated):
        assert not self.talk_over and pid == self.current_speaker
        self.transcript.append(dict(speaker=pid, text=text, truncated=truncated))
        self.seats[pid].sent += 1
        if len(self.transcript) >= 2 * cfg.max_messages:
            self.talk_over = True
        elif (cfg.allow_early_end and len(self.transcript) >= 2
              and all(END_MARK in u["text"] for u in self.transcript[-2:])):
            self.talk_over = self.ended_early = True

    def decide(self, pid, text, truncated):
        assert self.talk_over and self.decision_text[pid] is None
        self.decision_text[pid] = text
        self.choice[pid] = None if truncated else parse_decision(text)

    def payoffs(self):   # an unparseable decision counts as TAKE (nothing was given away)
        return PAYOFFS[tuple(c or TAKE for c in self.choice)]

    def reward(self, pid):
        if self.choice[pid] is None:
            return cfg.invalid_decision_reward
        p = self.payoffs()
        return float(sum(p) if cfg.pooled_reward else p[pid])

    def outcome(self):
        """GG = both give, TG = one takes and one gives, TT = both take, INV = someone's decision was unparseable."""
        c = self.choice
        if None in c:
            return "INV"
        return {0: "TT", 1: "TG", 2: "GG"}[c.count(GIVE)]

    def record(self, **extra):
        return dict(**extra, first_speaker=self.first, ended_early=self.ended_early, transcript=self.transcript,
                    decision_text=self.decision_text, choice=self.choice, outcome=self.outcome(),
                    payoffs=list(self.payoffs()), rewards=[self.reward(0), self.reward(1)])

## Model, LoRA, and batched lockstep rollouts

Every game advances one turn per round, and all pending turns across all games are sampled in one batched
`generate` call. Each seat's token sequence is built incrementally in Qwen's chat format, so the exact sampled
tokens are what gets trained (no re-tokenisation) and a whole game is one training sequence per seat.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

tok = AutoTokenizer.from_pretrained(cfg.model_name)
base = AutoModelForCausalLM.from_pretrained(cfg.model_name, torch_dtype=dtype, attn_implementation="sdpa").to(device)
model = get_peft_model(base, LoraConfig(
    r=cfg.lora_rank, lora_alpha=2 * cfg.lora_rank, lora_dropout=0.0, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=0.0)

IM_START, IM_END = "<|im_start|>", "<|im_end|>"
IM_END_ID = tok.convert_tokens_to_ids(IM_END)
STOP_IDS = sorted({IM_END_ID, tok.convert_tokens_to_ids("<|endoftext|>")})
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else STOP_IDS[-1]
NL = tok.encode("\n", add_special_tokens=False)


def enc(s):
    return tok.encode(s, add_special_tokens=False)


# Sanity check: our incremental format matches Qwen's own chat template.
_conv = [{"role": "system", "content": "S"}, {"role": "user", "content": "U"}, {"role": "assistant", "content": "A"},
         {"role": "user", "content": "V"}]
_ours = f"{IM_START}system\nS{IM_END}\n{IM_START}user\nU{IM_END}\n{IM_START}assistant\nA{IM_END}\n{IM_START}user\nV{IM_END}\n{IM_START}assistant\n"
assert tok.apply_chat_template(_conv, tokenize=False, add_generation_prompt=True) == _ours


def gen_config(max_new):
    # Qwen ships top_k=20/top_p=0.8/repetition_penalty; override for plain temperature sampling (on-policy RL).
    return GenerationConfig(do_sample=True, temperature=cfg.temperature, top_k=0, top_p=1.0, repetition_penalty=1.0,
                            max_new_tokens=max_new, eos_token_id=STOP_IDS, pad_token_id=PAD_ID)


@torch.no_grad()
def generate(prompts, max_new, use_adapter=True):
    """prompts: lists of token ids. Returns (new tokens up to and including the stop token, finished) per prompt."""
    model.eval()
    out = [None] * len(prompts)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))   # similar lengths share a batch
    for b in range(0, len(order), cfg.gen_batch_size):
        idx = order[b:b + cfg.gen_batch_size]
        L = max(len(prompts[i]) for i in idx)
        ids = torch.full((len(idx), L), PAD_ID, dtype=torch.long)
        attn = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):                                       # left padding
            ids[r, L - len(prompts[i]):] = torch.tensor(prompts[i])
            attn[r, L - len(prompts[i]):] = 1
        with (contextlib.nullcontext() if use_adapter else model.disable_adapter()):
            seqs = model.generate(input_ids=ids.to(device), attention_mask=attn.to(device),
                                  generation_config=gen_config(max_new))
        for r, i in enumerate(idx):
            new = seqs[r, L:].tolist()
            stop = next((k for k, t in enumerate(new) if t in STOP_IDS), None)
            out[i] = (new[:stop] + [IM_END_ID], True) if stop is not None else (new, False)
    return out


def play_games(n_games, policies=("train", "train"), first_offset=0):
    """Play n_games in lockstep. policies[pid] is 'train' (LoRA on) or 'base' (LoRA off) for seat pid.
    First speakers alternate, so they are balanced within every group."""
    games = [Game(first_speaker=(first_offset + i) % 2) for i in range(n_games)]
    while True:
        reqs = []
        for g in games:
            if g.done:
                continue
            reqs += [(g, g.current_speaker)] if not g.talk_over else [(g, p) for p in (0, 1) if g.decision_text[p] is None]
        if not reqs:
            return games
        prompts = []
        for g, p in reqs:
            s = g.seats[p]
            new = enc(f"{IM_START}system\n{SYSTEM}{IM_END}\n") if not s.tokens else []
            new += enc(f"{IM_START}user\n{g.user_turn(p)}{IM_END}\n{IM_START}assistant\n")
            s.tokens += new
            s.mask += [0] * len(new)
            prompts.append(list(s.tokens))
        results = [None] * len(reqs)
        for pol in set(policies):
            for deciding in (False, True):
                idx = [k for k, (g, p) in enumerate(reqs) if policies[p] == pol and g.talk_over == deciding]
                if idx:
                    res = generate([prompts[k] for k in idx],
                                   cfg.max_new_tokens_decision if deciding else cfg.max_new_tokens_msg, pol == "train")
                    for k, r in zip(idx, res):
                        results[k] = r
        for (g, p), (new, finished) in zip(reqs, results):
            s = g.seats[p]
            s.tokens += new                       # sampled tokens (incl. <|im_end|>) are trained
            s.mask += [1] * len(new)
            if not finished:                      # ran out of tokens: close the turn, don't train the patch
                s.tokens.append(IM_END_ID)
                s.mask.append(0)
            s.tokens += NL
            s.mask += [0] * len(NL)
            s.turns += 1
            text = tok.decode(new, skip_special_tokens=True).strip()
            (g.decide if g.talk_over else g.post)(p, text, not finished)


def show_game(g):
    for u in g.transcript:
        print(f"[P{u['speaker']}{' (cut off)' if u['truncated'] else ''}] {u['text']}\n")
    for p in (0, 1):
        print(f"--- P{p} decides {g.choice[p]}: {g.decision_text[p]!r}")
    print(f"outcome {g.outcome()}, payoffs {g.payoffs()}, rewards {[g.reward(0), g.reward(1)]}")


t0 = time.time()
_demo = play_games(2)
print(f"2 untrained games in {time.time() - t0:.1f}s\n")
show_game(_demo[0])

## RL: GRPO-style policy gradient

Advantages are centred over each group of 8 games (16 trajectories). Both seats of a game share the pooled
reward, so the learning signal comes from comparing *games* within a group. The loss is
`-advantage x log p(token)` summed over every token the model sampled, divided by the batch's sampled-token
count. One gradient step per batch of fresh games, so no importance ratio or clipping is needed.

In [ ]:
backbone, lm_head = model.get_base_model().model, model.get_base_model().lm_head


def train_step(games):
    trajs = []                                     # (tokens, mask, advantage)
    G = cfg.games_per_group
    for i in range(0, len(games), G):
        grp = [(g, p) for g in games[i:i + G] for p in (0, 1)]
        r = torch.tensor([g.reward(p) for g, p in grp])
        adv = r - r.mean()
        if cfg.normalize_adv:
            adv = adv / (r.std() + 1e-6)
        trajs += [(g.seats[p].tokens, g.seats[p].mask, a.item()) for (g, p), a in zip(grp, adv) if abs(a) > 1e-6]
    if not trajs:
        return dict(loss=0.0, grad_norm=0.0, n_traj=0)
    n_tok = sum(sum(m) for _, m, _ in trajs)
    trajs.sort(key=lambda t: len(t[0]))
    model.train()
    opt.zero_grad(set_to_none=True)
    total = 0.0
    for b in range(0, len(trajs), cfg.micro_batch):
        mb = trajs[b:b + cfg.micro_batch]
        L = max(len(t) for t, _, _ in mb)
        ids = torch.full((len(mb), L), PAD_ID, dtype=torch.long)
        attn = torch.zeros((len(mb), L), dtype=torch.long)
        mask = torch.zeros((len(mb), L), dtype=torch.bool)
        for r, (t, m, _) in enumerate(mb):          # right padding
            ids[r, :len(t)], attn[r, :len(t)], mask[r, :len(t)] = torch.tensor(t), 1, torch.tensor(m, dtype=torch.bool)
        ids, attn, mask = ids.to(device), attn.to(device), mask.to(device)
        adv = torch.tensor([a for _, _, a in mb], device=device)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=bf16):
            h = backbone(input_ids=ids, attention_mask=attn).last_hidden_state
            tgt = mask[:, 1:]                        # position t predicts token t+1
            logits = lm_head(h[:, :-1][tgt])         # only the sampled positions: saves a [B, L, 152k] tensor
        logp = -F.cross_entropy(logits.float(), ids[:, 1:][tgt], reduction="none")
        loss = -(adv[:, None].expand_as(tgt)[tgt] * logp).sum() / n_tok
        loss.backward()
        total += loss.item()
    gn = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.max_grad_norm)
    opt.step()
    return dict(loss=total, grad_norm=gn.item(), n_traj=len(trajs))


def game_stats(games):
    oc = Counter(g.outcome() for g in games)
    n = len(games)
    choices = [c for g in games for c in g.choice]
    return {
        **{f"frac_{k}": oc[k] / n for k in ("TG", "GG", "TT", "INV")},
        "give_rate": choices.count(GIVE) / len(choices),
        "invalid_rate": choices.count(None) / len(choices),
        "mean_reward": sum(g.reward(p) for g in games for p in (0, 1)) / (2 * n),
        "mean_pooled_payoff": sum(sum(g.payoffs()) for g in games) / n,
        "mean_messages": sum(len(g.transcript) for g in games) / n,
        "ended_early": sum(g.ended_early for g in games) / n,
        "truncated_msgs": sum(u["truncated"] for g in games for u in g.transcript) / max(1, sum(len(g.transcript) for g in games)),
    }

In [ ]:
history = []
games_file = os.path.join(cfg.log_path, "games.jsonl")
n_games = cfg.groups_per_batch * cfg.games_per_group
print(f"{n_games} games/step, {cfg.steps} steps, reward={'pooled' if cfg.pooled_reward else 'individual'}")
print(f"{'step':>4} {'TG':>5} {'GG':>5} {'TT':>5} {'INV':>5} {'give':>5} {'reward':>6} {'msgs':>4} {'gnorm':>6} {'sec':>5}")
for step in range(cfg.steps):
    t0 = time.time()
    games = play_games(n_games, first_offset=step)
    t_roll = time.time() - t0
    info = train_step(games)
    st = {"step": step, **game_stats(games), **info, "t_rollout": t_roll, "t_step": time.time() - t0}
    history.append(st)
    with open(games_file, "a") as f:
        for i, g in enumerate(games):
            f.write(json.dumps(g.record(step=step, game=i)) + "\n")
    print(f"{step:>4} {st['frac_TG']:>5.2f} {st['frac_GG']:>5.2f} {st['frac_TT']:>5.2f} {st['frac_INV']:>5.2f} "
          f"{st['give_rate']:>5.2f} {st['mean_reward']:>6.2f} {st['mean_messages']:>4.1f} {st['grad_norm']:>6.3f} "
          f"{st['t_step']:>5.0f}")
json.dump(history, open(os.path.join(cfg.log_path, "history.json"), "w"), indent=1)
model.save_pretrained(os.path.join(cfg.log_path, "adapter"))
print("saved to", cfg.log_path)

## Training curves

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
series = [("frac_TG", "Take/Give", "#2a78d6"), ("frac_GG", "Give/Give", "#1baf7a"),
          ("frac_TT", "Take/Take", "#eb6834"), ("frac_INV", "invalid", "#eda100")]
for key, label, color in series:
    ax1.plot(steps, [h[key] for h in history], color=color, lw=2, label=label)
ax1.set(title="Game outcomes per step (fraction of games)", xlabel="step", ylim=(0, 1))
ax1.legend(frameon=False, loc="upper left")
ax2.plot(steps, [h["mean_pooled_payoff"] for h in history], color="#2a78d6", lw=2)
ax2.set(title="Mean pooled payoff per game (max 6)", xlabel="step", ylim=(0, 6.2))
for ax in (ax1, ax2):
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## A few games from the last training step

In [ ]:
for g in games[:3]:
    show_game(g)
    print("=" * 100)

## Evaluation: base vs base, trained vs trained, trained vs base

Fresh games with the same sampling settings. In the mixed matchup seat 0 is always the trained model, so the
per-seat give rates show who sacrifices.

In [ ]:
def eval_matchup(name, policies):
    gs = play_games(cfg.eval_games, policies)
    st = game_stats(gs)
    give = [sum(g.choice[p] == GIVE for g in gs) / len(gs) for p in (0, 1)]
    print(f"{name:<20} TG {st['frac_TG']:.2f}  GG {st['frac_GG']:.2f}  TT {st['frac_TT']:.2f}  INV {st['frac_INV']:.2f}"
          f"  | pooled {st['mean_pooled_payoff']:.2f}  | give rate seat0 {give[0]:.2f} seat1 {give[1]:.2f}")
    return gs


eval_games = {
    "base vs base": eval_matchup("base vs base", ("base", "base")),
    "trained vs trained": eval_matchup("trained vs trained", ("train", "train")),
    "trained vs base": eval_matchup("trained vs base", ("train", "base")),
}

In [ ]:
# Optional: keep the run. Mount Drive and copy the run folder (adapter, games.jsonl, history.json).
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r {cfg.log_path} /content/drive/MyDrive/